In [1]:
import numpy as np
import pandas as pd

# Завдання 1. Побудова "брудного" набору даних для Варіанта 2 (Львів)
raw_data = [
    {"id": 1, "місто": "Львів", "ціна": 720.0, "кількість": 4},
    {"id": 2, "місто": " львів ", "ціна": "720 грн", "кількість": np.nan},  # пропуск у кількості + пробіли в місті + ціна як рядок
    {"id": 3, "місто": "ЛЬВІВ", "ціна": 720.0, "кількість": 2},            # місто у верхньому регістрі
    {"id": 3, "місто": "ЛЬВІВ", "ціна": 720.0, "кількість": 2},            # дублікат рядка id=3
    {"id": 4, "місто": "Львів", "ціна": "720 грн", "кількість": 5},        # ціна як рядок із суфіксом
    {"id": 5, "місто": "Львів", "ціна": 7200.0, "кількість": 3},           # викид у ціні (в 10 разів більше за 720)
    {"id": 6, "місто": "Львів", "ціна": 720.0, "кількість": np.nan},       # другий пропуск у кількості
    {"id": 7, "місто": "Львів", "ціна": 720.0, "кількість": 4}
]

df_dirty = pd.DataFrame(raw_data)
print("Вихідний 'брудний' набір даних:")
display(df_dirty)

Вихідний 'брудний' набір даних:


,id,місто,ціна,кількість
0,1,Львів,720.0,4.0
1,2,львів,720 грн,NaN
2,3,ЛЬВІВ,720.0,2.0
3,3,ЛЬВІВ,720.0,2.0
4,4,Львів,720 грн,5.0
5,5,Львів,7200.0,3.0
6,6,Львів,720.0,NaN
7,7,Львів,720.0,4.0


In [2]:
# Завдання 2.1. Визначення кількості пропусків у кожному стовпці
print("Кількість пропусків до обробки:")
print(df_dirty.isna().sum())

# Завдання 2.2. Заповнення пропусків у стовпці "кількість" медіаною
median_qty = df_dirty["кількість"].median()
df_dirty["кількість"] = df_dirty["кількість"].fillna(median_qty)

print(f"\nМедіана кількості, якою заповнили пропуски: {median_qty}")
print("Кількість пропусків після заповнення:")
print(df_dirty.isna().sum())

Кількість пропусків до обробки:
id           0
місто        0
ціна         0
кількість    2
dtype: int64

Медіана кількості, якою заповнили пропуски: 3.5
Кількість пропусків після заповнення:
id           0
місто        0
ціна         0
кількість    0
dtype: int64


**Пояснення до вибору медіани замість середнього:**
У нашому наборі всього 8 спостережень, і частина значень може бути чутливою до невеликих коливань. Медіана є стійкішою (робастною) характеристикою до малих вибірок та можливих аномалій порівняно із середнім арифметичним, яке через малу кількість рядків могло б спотворитися або дати неціле значення там, де кількість товару має бути цілою.

In [3]:
# Завдання 3.1. Перевірка дублікатів без subset (повний збіг) та з subset за "id"
full_duplicates_count = df_dirty.duplicated().sum()
id_duplicates_count = df_dirty.duplicated(subset=["id"]).sum()

print(f"Повних дублікатів рядків (без subset): {full_duplicates_count}")
print(f"Дублікатів за унікальним ідентифікатором (subset=['id']): {id_duplicates_count}")

# Завдання 3.2. Видалення дублікатів за subset (зберігаємо перший варіант)
df_clean = df_dirty.drop_duplicates(subset=["id"], keep="first").copy()

print(f"\nКількість рядків після видалення дублікатів: len(df_clean) = {len(df_clean)}")
display(df_clean)

Повних дублікатів рядків (без subset): 1
Дублікатів за унікальним ідентифікатором (subset=['id']): 1

Кількість рядків після видалення дублікатів: len(df_clean) = 7


,id,місто,ціна,кількість
0,1,Львів,720.0,4.0
1,2,львів,720 грн,3.5
2,3,ЛЬВІВ,720.0,2.0
4,4,Львів,720 грн,5.0
5,5,Львів,7200.0,3.0
6,6,Львів,720.0,3.5
7,7,Львів,720.0,4.0


In [4]:
# Завдання 4.1. Приведення стовпця "ціна" до числового типу (float)
# Замінюємо рядок " грн" на порожній рядок, перетворюємо через pd.to_numeric з errors="coerce"
df_clean["ціна"] = df_clean["ціна"].astype(str).str.replace(" грн", "", regex=False)
df_clean["ціна"] = pd.to_numeric(df_clean["ціна"], errors="coerce")

# Перевірка на появу нових NaN після конвертації
print("Кількість NaN у стовпці 'ціна' після перетворення:", df_clean["ціна"].isna().sum())

# Завдання 4.2. Узгодження категорій у стовпці "місто"
print("Унікальні значення міста ДО узгодження:", df_clean["місто"].unique())

# Прибираємо зайві пробіли, переводимо в нижній регістр і робимо першу літеру великою
df_clean["місто"] = df_clean["місто"].str.strip().str.lower().str.capitalize()

print("Унікальні значення міста ПІСЛЯ узгодження:", df_clean["місто"].unique())
display(df_clean)

Кількість NaN у стовпці 'ціна' після перетворення: 0
Унікальні значення міста ДО узгодження: <StringArray>
['Львів', ' львів ', 'ЛЬВІВ']
Length: 3, dtype: str
Унікальні значення міста ПІСЛЯ узгодження: <StringArray>
['Львів']
Length: 1, dtype: str


,id,місто,ціна,кількість
0,1,Львів,720.0,4.0
1,2,Львів,720.0,3.5
2,3,Львів,720.0,2.0
4,4,Львів,720.0,5.0
5,5,Львів,7200.0,3.0
6,6,Львів,720.0,3.5
7,7,Львів,720.0,4.0


In [5]:
# Завдання 5.1. Пошук меж IQR для стовпця "ціна"
q1, q3 = df_clean["ціна"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print(f"Нижня межа = {lower_bound}, Верхня межа = {upper_bound}")

# Виявлення викидів
outliers = df_clean[(df_clean["ціна"] < lower_bound) | (df_clean["ціна"] > upper_bound)]
print("\nЗнайдені викиди:")
display(outliers)

Q1 = 720.0, Q3 = 720.0, IQR = 0.0
Нижня межа = 720.0, Верхня межа = 720.0

Знайдені викиди:


,id,місто,ціна,кількість
5,5,Львів,7200.0,3.0


### Письмові відповіді на контрольні запитання

1. **Щодо вибору рішення для викиду ціни (7200 замість 720):**
   Для цього конкретного значення (яке рівно в 10 разів перевищує базову ціну) доцільно обрати рішення **вважати його помилкою вводу (технічним збоєм, наприклад, зайвим нулем)**, а не реальним оптовим замовленням. Оскільки наш умовний інтернет-магазин продає стандартний роздрібний товар, а замовлення виглядають як одиничні покупки, випадковий зайвий символ під час введенща є найімовірнішою причиною. У такому разі його слід скоригувати (поділити на 10), а не видаляти чи залишати без змін.

2. **Чому заповнення пропуску середнім (а не медіаною) може згодом спотворити стандартне відхилення?**
   Заповнення пропусків константою (зокрема середнім значенням) штучно створює скупчення даних у тій самій точці та зменшує загальну варіативність вибірки. Оскільки стандартне відхилення вимірює розкид відносно середнього, додавання однакових середніх значень зменшує середньоквадратичне відхилення, що спотворює подальший статистичний аналіз.

3. **У чому різниця між duplicated() без subset і з subset?**
   Метод `duplicated()` без аргументів шукає рядки, у яких повністю збігаються абсолютно всі стовпці. Якщо ж передати параметр `subset=["id"]`, перевірка здійснюватиметься виключно за вказаним полем (або полями), що дозволяє знаходити дублікати за бізнес-логікою (наприклад, той самий клієнт або замовлення під одним ID), навіть якщо інші другорядні дані в рядках відрізняються.

4. **Чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?**
   Критерій IQR є лише формальним статистичним інструментом для *позначення* підозрілих аномалій, а не автоматичним правилом для знищення даних. Поза межами IQR можуть опинятися цілком реальні, хоч і рідкісні події (наприклад, екстремально високі зарплати керівництва або великі оптові замовлення), і їх бездумне видалення знищить важливу цінну інформацію про хвіст розподілу.